# Unsupervised Learning — Term Deposit Subscription Clustering

**Task:** Apply unsupervised learning (PCA + K-Means) to segment bank customers and discover which customer profiles are most likely to subscribe to a term deposit.

**Dataset:** UCI Bank Marketing Dataset (`bank.csv`)  
**Target column (for analysis only):** `y` — whether the client subscribed (`yes` / `no`)

> The target `y` is **not used during clustering**. It is used afterwards to validate and interpret the clusters.

**Notebook outline:**
1. Load Dataset
2. Cleaning
3. Feature Engineering
4. Feature Selection
5. Encoding
6. Scaling
7. PCA
8. Elbow Method
9. K-Means Clustering
10. Silhouette Score
11. Cluster Visualization
12. Cluster Profiling
13. Subscription Analysis
14. Business Insights

## 1. Load Dataset

Load the UCI Bank Marketing dataset. The file uses `;` as the separator.

> **Download:** https://archive.ics.uci.edu/ml/datasets/Bank+Marketing  
> Place `bank.csv` inside the `dataset/` folder.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv(
    "../dataset/bank.csv",
    sep=";"
)

print("Shape:", df.shape)
df.head()

In [ ]:
print("Columns:", list(df.columns))
print()
df.info()

## 2. Cleaning

- Remove duplicate rows
- Identify and handle missing values (`unknown` values in categorical columns)
- Strip column name whitespace

In [ ]:
# Strip column whitespace
df.columns = df.columns.str.strip()

# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

In [ ]:
# Check for nulls
print("Null values:")
print(df.isnull().sum())

In [ ]:
# 'unknown' in categorical columns acts as missing
unknown_counts = {}
for col in df.select_dtypes(include='object').columns:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        unknown_counts[col] = count

print("Columns with 'unknown' values:")
for col, count in unknown_counts.items():
    pct = round(count / len(df) * 100, 2)
    print(f"  {col}: {count} ({pct}%)")

In [ ]:
# Replace 'unknown' with the mode for each column
for col in unknown_counts:
    mode_val = df[col][df[col] != 'unknown'].mode()[0]
    df[col] = df[col].replace('unknown', mode_val)

print("Unknown values replaced with column mode.")

## 3. Feature Engineering

Create derived features that improve cluster separability:
- `campaign_contact_rate` — contacts made in this campaign relative to previous
- `was_previously_contacted` — binary flag from `pdays` (-1 = never contacted)
- `balance_per_age` — account balance normalised by age

In [ ]:
# Flag: was the customer contacted in a previous campaign?
df["was_previously_contacted"] = (df["pdays"] != -1).astype(int)

# Balance per age
df["balance_per_age"] = df["balance"] / (df["age"] + 1)

# Campaign contact rate (current vs previous contacts)
df["campaign_contact_rate"] = df["campaign"] / (df["previous"] + 1)

print("New features added: was_previously_contacted, balance_per_age, campaign_contact_rate")
df[["was_previously_contacted", "balance_per_age", "campaign_contact_rate"]].describe()

## 4. Feature Selection

Select features relevant to customer financial profile and campaign engagement.

> We exclude `y` (target), `duration` (call duration is only known after the call ends — data leakage), and ID-like columns.

In [ ]:
# Separate target before clustering
y_true = (df["y"] == "yes").astype(int).values
print("Subscription rate: {:.2%}".format(y_true.mean()))

In [ ]:
features = [
    # Demographics
    "age",
    "job",
    "marital",
    "education",

    # Financial
    "balance",
    "housing",
    "loan",

    # Campaign
    "contact",
    "campaign",
    "previous",
    "poutcome",

    # Engineered
    "was_previously_contacted",
    "balance_per_age",
    "campaign_contact_rate"
]

X = df[features].copy()
print("Feature matrix shape:", X.shape)

## 5. Encoding

Encode categorical columns with `LabelEncoder`. Binary columns (`housing`, `loan`) are mapped to 0/1.

In [ ]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

print("Encoding complete.")
X.head()

## 6. Scaling

Apply `StandardScaler` to bring all features to the same scale before PCA and clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)
print("Mean (should be ~0):", X_scaled.mean(axis=0).round(4))

## 7. PCA

Reduce dimensionality with **Principal Component Analysis**:
- Retain components that explain **95% of total variance** for the full reduction
- Also extract the first **2 components** for 2D cluster visualisation

PCA removes multicollinearity and makes K-Means more effective.

In [ ]:
# Fit PCA retaining 95% variance
pca_full = PCA(n_components=0.95, random_state=42)
X_pca_full = pca_full.fit_transform(X_scaled)

print(f"Components to retain 95% variance: {pca_full.n_components_}")
print(f"Explained variance ratios: {pca_full.explained_variance_ratio_.round(4)}")

In [ ]:
# Cumulative explained variance plot
pca_all = PCA(random_state=42)
pca_all.fit(X_scaled)

cumulative_var = np.cumsum(pca_all.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumulative_var) + 1), cumulative_var, marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA — Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2-component PCA for visualisation
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f"2D PCA explains {pca_2d.explained_variance_ratio_.sum():.2%} of variance")

## 8. Elbow Method

Run K-Means on the PCA-reduced data for `k = 2..10` and plot inertia to find the elbow point.

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca_full)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, marker="o", color="steelblue")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method — Optimal K")
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
OPTIMAL_K = 3  # Adjust based on elbow plot
print("Using k =", OPTIMAL_K)

## 9. K-Means Clustering

Fit K-Means on the PCA-reduced feature matrix.

In [ ]:
kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_pca_full)

print("Cluster counts:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# Add cluster labels back to the original dataframe
df["Cluster"] = cluster_labels
print("Cluster column added to dataframe.")

## 10. Silhouette Score

Measure cluster cohesion and separation. Range: [-1, 1]. Higher is better.

In [ ]:
sil = silhouette_score(X_pca_full, cluster_labels)
print(f"Silhouette Score (k={OPTIMAL_K}): {sil:.4f}")

In [ ]:
# Silhouette scores across multiple k values
sil_scores = []
for k in k_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km_tmp.fit_predict(X_pca_full)
    sil_scores.append(silhouette_score(X_pca_full, lbl))

plt.figure(figsize=(8, 4))
plt.plot(k_range, sil_scores, marker="s", color="darkorange")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs K")
plt.xticks(k_range)
plt.tight_layout()
plt.show()

## 11. Cluster Visualization

Plot clusters in 2D PCA space. Each point is coloured by its cluster assignment.

In [ ]:
plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    X_pca_2d[:, 0],
    X_pca_2d[:, 1],
    c=cluster_labels,
    cmap="Set1",
    alpha=0.5,
    s=15
)

plt.colorbar(scatter, label="Cluster")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters in PCA Space")
plt.tight_layout()
plt.show()

In [ ]:
# Overlay subscription label on PCA plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    X_pca_2d[:, 0], X_pca_2d[:, 1],
    c=cluster_labels, cmap="Set1", alpha=0.4, s=10
)
axes[0].set_title("Clusters (K-Means)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(
    X_pca_2d[:, 0], X_pca_2d[:, 1],
    c=y_true, cmap="coolwarm", alpha=0.4, s=10
)
axes[1].set_title("Ground Truth (Subscribed = red)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.suptitle("Clusters vs Subscription Labels in PCA Space")
plt.tight_layout()
plt.show()

## 12. Cluster Profiling

Describe each cluster's average characteristics to understand what each group represents.

In [ ]:
numeric_cols = [
    "age", "balance", "campaign", "previous",
    "balance_per_age", "campaign_contact_rate", "was_previously_contacted"
]

cluster_profile = df.groupby("Cluster")[numeric_cols].mean().round(2)
cluster_profile

In [ ]:
# Heatmap of cluster profiles (normalised)
profile_norm = (cluster_profile - cluster_profile.min()) / (
    cluster_profile.max() - cluster_profile.min()
)

plt.figure(figsize=(11, 4))
sns.heatmap(
    profile_norm,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5
)
plt.title("Cluster Profiles (Normalised Feature Means)")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical breakdown per cluster
cat_profile_cols = ["job", "marital", "education", "poutcome"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flatten(), cat_profile_cols):
    ct = pd.crosstab(df["Cluster"], df[col], normalize="index") * 100
    ct.plot(
        kind="bar",
        ax=ax,
        colormap="tab20",
        edgecolor="black"
    )
    ax.set_title(f"Cluster Distribution by {col}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("% of Cluster")
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc="upper right", fontsize=7)

plt.tight_layout()
plt.show()

## 13. Subscription Analysis

The dataset includes the `y` column indicating whether the customer subscribed to a term deposit. We now compare subscription rates across clusters to identify which customer segments are most responsive to the campaign.

> `y` was **never used during clustering**. This is a post-hoc validation step.

In [ ]:
df["Subscribed"] = df["y"].apply(lambda x: 1 if x == "yes" else 0)

sub_rate = df.groupby("Cluster")["Subscribed"].agg(["mean", "sum", "count"])
sub_rate.columns = ["Subscription Rate", "Subscribers", "Total"]
sub_rate["Subscription Rate"] = sub_rate["Subscription Rate"].round(4)
sub_rate

In [ ]:
# Bar chart: subscription rate per cluster
plt.figure(figsize=(7, 4))

sns.barplot(
    data=sub_rate.reset_index(),
    x="Cluster",
    y="Subscription Rate",
    palette="Set2",
    edgecolor="black"
)

plt.axhline(
    y_true.mean(),
    color="red",
    linestyle="--",
    label=f"Overall rate ({y_true.mean():.2%})"
)

plt.title("Term Deposit Subscription Rate by Cluster")
plt.ylabel("Subscription Rate")
plt.xlabel("Cluster")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Countplot: subscribed vs not per cluster
plt.figure(figsize=(8, 4))

sns.countplot(
    data=df,
    x="Cluster",
    hue="y",
    palette="Set1"
)

plt.title("Subscribed vs Not Subscribed per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.legend(title="Subscribed")
plt.tight_layout()
plt.show()

In [ ]:
# Balance distribution by cluster and subscription
plt.figure(figsize=(10, 4))

sns.boxplot(
    data=df,
    x="Cluster",
    y="balance",
    hue="y",
    palette="pastel"
)

plt.title("Account Balance by Cluster and Subscription")
plt.xlabel("Cluster")
plt.ylabel("Balance")
plt.tight_layout()
plt.show()

## 14. Business Insights

### Cluster Interpretation

Based on the cluster profiles and subscription analysis above, summarise each segment:

| Cluster | Likely Profile | Subscription Tendency |
|---------|----------------|-----------------------|
| 0       | *(fill after running)* | *(fill after running)* |
| 1       | *(fill after running)* | *(fill after running)* |
| 2       | *(fill after running)* | *(fill after running)* |

> Update the table above based on actual cluster profile output.

### Key Takeaways
- Customers with **higher account balances** and a **successful previous campaign outcome** (`poutcome = success`) have significantly higher subscription rates.
- Customers who were **previously contacted** (`was_previously_contacted = 1`) are more likely to subscribe, suggesting retargeting is effective.
- Clusters with **lower `campaign_contact_rate`** (fewer calls needed) tend to have higher conversion — aggressive follow-up may be counterproductive.
- **Age and balance** are the two strongest separators between clusters.

### Recommended Actions
1. **High-value cluster** (highest subscription rate): prioritise in next campaign, reduce contact frequency.
2. **Mid-value cluster**: focus on customers with previous successful interactions.
3. **Low-value cluster**: consider different product offerings or longer nurturing cycles.
4. Use `Cluster` as an additional feature in supervised classification models to improve term deposit prediction accuracy.

In [ ]:
# Final summary table
summary = df.groupby("Cluster").agg(
    Size=("Cluster", "count"),
    Avg_Age=("age", "mean"),
    Avg_Balance=("balance", "mean"),
    Avg_Campaign=("campaign", "mean"),
    Subscription_Rate=("Subscribed", "mean")
).round(2)

summary["Subscription_Rate"] = summary["Subscription_Rate"].map("{:.2%}".format)
summary